In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Import usual modules
import pandas as pd
import csv
import math
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import openpyxl
import datetime
from scipy.stats import lognorm
import re
import string
from bs4 import BeautifulSoup
import requests
import unicodedata # for removing accented characters
import icecream as ic

import pdfplumber


In [3]:
os.getcwd()
#os.chdir("/Users/veesheen/Desktop")

'/Users/veesheenyuen/code/veeyuen/SAA'

# 2026 NSG Consolidated PDF

In [5]:
import os
import re
import pandas as pd
import pdfplumber

# =========================================================
# PATCHED VERSION FOR 2026 CONSOLIDATED BORDER-BASED PDF
# =========================================================

os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/NSG/2026/Consolidated')
file = "/Users/veesheenyuen/Desktop/DataScience/SAA/NSG/2026/Consolidated/sectrack_result_2026a.pdf"

# ---------------------------------------------------------
# Regex / constants
# ---------------------------------------------------------
session_pattern = r'\d\d\-\d\d'
wind_pattern = r'(H[0-9]|H[0-9]{2})\s*:\s*([+\-]?\d\.\d)'
result_pattern = r'(?:\d{1,2}:\d{2}\.\d{2}|\d+\.\d{2})'
status_codes = {'NM', 'DNS', 'DNF', 'DQ'}

FINAL_SCHEMA = [
    'FIRST_NAME', 'LAST_NAME', 'OTHER_NAME', 'NAME', 'RANK', 'TAG_ID', 'TEAM', 'SEED', 'RESULT', 'QUALIFICATION',
    'HEAT', 'LANE', 'WIND', 'EVENT', 'DIVISION', 'STAGE', 'POINTS', 'AGE', 'GENDER', 'UNIQUE_ID', 'NATIONALITY',
    'DICT_RESULTS', 'YEAR', 'DATE', 'COMPETITION', 'REGION', 'DOB', 'GROUP', 'CATEGORY_EVENT', 'ATHLETE_ID',
    'SOURCE', 'REMARKS', 'TIMESTAMP', 'VENUE', 'SUB_EVENT', 'SESSION', 'EVENT_CLASS', 'DISTANCE', 'HOST_CITY',
    'RX_TIME'
]

SOURCE_URL = 'https://nsg.rjcat.com/sectrack_2026/ss_home.php?mypage=zz_announce.htm'


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def clean_cell(x):
    if x is None:
        return ''
    x = str(x).replace('\n', ' ').replace('\xa0', ' ').strip()
    x = re.sub(r'\s+', ' ', x)
    return x


def infer_category_event(event_name: str) -> str:
    if not event_name:
        return ''

    e = event_name.lower()

    if 'relay' in e:
        return 'Relay'
    if 'hurdle' in e:
        return 'Hurdles'
    if 'walk' in e:
        return 'Walk'
    if 's/c' in e or 'steeple' in e:
        return 'Steeple'
    if 'jump' in e or 'vault' in e:
        return 'Jump'
    if any(x in e for x in ['javelin', 'shot put', 'shot', 'discus', 'hammer', 'throw']):
        return 'Throw'
    if any(x in e for x in ['5000', '3000']):
        return 'Long'
    if any(x in e for x in ['1500', '2000', '800']):
        return 'Mid'
    if any(x in e for x in ['100', '200', '400']):
        return 'Sprint'

    return ''


def parse_page_metadata(text: str, previous_meta: dict) -> dict:
    """
    Parse event/date/stage/division metadata from page text.
    Falls back to previous metadata where needed.
    """
    meta = previous_meta.copy()
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    # Date
    date_match = re.search(r'\b\d{2}-\d{2}-\d{4}\b', text)
    if date_match:
        meta['DATE'] = date_match.group(0)

    # Event line
    event_line = None
    for line in lines:
        if line.startswith('Event '):
            event_line = line
            break

    if event_line:
        m = re.search(r'^Event\s+(.*?)\s+(\d{2}-\d{2})\s*$', event_line)
        if m:
            raw_event = m.group(1).strip()
            session = m.group(2).strip()

            # Detect division and gender
            div_match = re.search(r'\(([A-Z])-(Boys|Girls)\)', raw_event, re.I)

            gender = meta.get('GENDER', '')
            div = meta.get('DIVISION', '')

            if div_match:
                div = div_match.group(1).upper()
                gender = 'Male' if div_match.group(2).lower() == 'boys' else 'Female'
            else:
                if re.search(r'boys', raw_event, re.I):
                    gender = 'Male'
                elif re.search(r'girls', raw_event, re.I):
                    gender = 'Female'

            # Extract measurement-only EVENT_CLASS directly from raw_event
            measurement_match = re.search(
                r'\((\d+(?:\.\d+)?\s*(?:kg|g|m))\)',
                raw_event,
                re.I
            )
            event_class = measurement_match.group(1).strip() if measurement_match else ''

            # Remove division token
            event_text = re.sub(
                r'\(([A-Z])-(Boys|Girls)\)\s*$',
                '',
                raw_event,
                flags=re.I
            ).strip()

            # Handle Para
            sub_event = ''
            if re.search(r'\(Para\)', event_text, re.I):
                sub_event = 'Para'
                event_text = re.sub(r'\(Para\)', '', event_text, flags=re.I).strip()

            # Remove measurement class from event name
            event_name = re.sub(
                r'\(\s*\d+(?:\.\d+)?\s*(?:kg|g|m)\s*\)',
                '',
                event_text,
                flags=re.I
            ).strip()

            # Remove leading ': ' if present
            event_name = re.sub(r'^:\s*', '', event_name).strip()

            meta['EVENT'] = event_name
            meta['SESSION'] = 'S' + session
            meta['DIVISION'] = div
            meta['GENDER'] = gender
            meta['SUB_EVENT'] = sub_event
            meta['EVENT_CLASS'] = event_class
            meta['CATEGORY_EVENT'] = infer_category_event(event_name)

    # Stage
    stage = meta.get('STAGE', '')
    if re.search(r':\s*Final\b', text, re.I):
        stage = 'Final'
    elif re.search(r':\s*Heat\b', text, re.I):
        stage = 'Heats'
    elif re.search(r':\s*SF\b', text, re.I):
        stage = 'SF'
    meta['STAGE'] = stage

    return meta


def extract_explicit_wind_dict(text: str):
    """
    Extract wind only from the explicit W/Gauge section for the current event.
    Returns (wind_dict, has_explicit_wind_values).
    """
    m = re.search(
        r'W/Gauge\s*:(.*?)(?:\nPos\s+Tag\s+Competitor|\Z)',
        text,
        flags=re.I | re.S
    )

    if not m:
        return {}, False

    wind_block = m.group(1).strip()

    matches = re.findall(
        r'(H\d{1,2})\s*:\s*([+\-]?\d\.\d)',
        wind_block,
        flags=re.I
    )

    wind_dict = {heat.upper(): val for heat, val in matches}
    has_explicit_wind_values = len(wind_dict) > 0

    return wind_dict, has_explicit_wind_values


def extract_table_with_fallback(page):
    settings_list = [
        {
            "vertical_strategy": "lines",
            "horizontal_strategy": "lines",
            "intersection_tolerance": 5,
            "snap_tolerance": 3,
            "join_tolerance": 3,
            "edge_min_length": 15
        },
        {
            "vertical_strategy": "lines_strict",
            "horizontal_strategy": "lines",
            "intersection_tolerance": 5,
            "snap_tolerance": 3,
            "join_tolerance": 3,
            "edge_min_length": 15
        },
        {
            "vertical_strategy": "text",
            "horizontal_strategy": "text"
        }
    ]

    for settings in settings_list:
        try:
            table = page.extract_table(settings)
            if table:
                return table
        except Exception:
            pass

    return None


def is_junk_row(cells):
    joined = ' '.join([c for c in cells if c]).strip()

    if not joined:
        return True

    junk_patterns = [
        r'~~~\s*End of Listing\s*~~~',
        r'DataQueryFooter',
        r'^Pos\s+Tag\s+Competitor',
        r'^W/Gauge',
        r'^Reason\s*:',
        r'^Event\s',
        r'^Singapore Schools Sports Council$',
        r'^National School Games',
        r'^Result List$',
        r'^Page\s+\d+$',
        r'^\d{2}-\d{2}-\d{4}$',
        r'^\d{2}:\d{2}\s*(AM|PM|am|pm)$',
        r'^Type\s*:'
    ]

    for pat in junk_patterns:
        if re.search(pat, joined):
            return True

    return False


def split_name_team(body: str):
    """
    Split a flattened body string into NAME and TEAM by detecting likely team suffixes.
    """
    body = re.sub(r'\s+', ' ', body).strip()
    parts = body.split()

    if len(parts) < 2:
        return body, ''

    team_hints = [
        'School', 'Schools', 'Institution', 'College', 'Convent', 'Sec',
        'Secondary', 'Junior', 'High', 'Sports', 'Methodist', 'Catholic',
        'International', 'Independent', 'Girls', 'Boys', 'CHIJ', 'ACS',
        'Anglo-Chinese', 'Victoria', 'Raffles', 'Nanyang', 'Crescent',
        'Dunman', 'Temasek', 'National', 'Eunoia', 'Hwa', 'Singapore',
        'Tanjong', 'Bedok', 'Bukit', 'North', 'Yishun', 'Jurong', 'Nan',
        'Cedar', 'St.', 'Presbyterian', 'Meridian', 'Chinese', 'Orchard',
        'Pathlight', 'Unity', 'Junyuan', 'Hai', 'Sing', 'Ahmad', 'Ibrahim',
        'Woodlands', 'Westwood', 'River', 'Valley', 'Yuan', 'Ching', 'Maris',
        'Stella', 'Whitley', 'Bendemeer', 'Deyi', 'Edgefield', 'Peicai'
    ]

    for i in range(len(parts) - 1, 0, -1):
        suffix = ' '.join(parts[i:])
        if any(hint in suffix for hint in team_hints):
            name = ' '.join(parts[:i]).strip()
            team = suffix.strip()
            if name and team:
                return name, team

    return ' '.join(parts[:-1]).strip(), parts[-1].strip()


def parse_tail_from_text(tail_text: str):
    """
    Parse trailing fields from right to left.
    """
    tail_text = re.sub(r'\s+', ' ', tail_text).strip()
    if not tail_text:
        return None

    # Status rows: heat lane NM|DNS|DNF|DQ
    m = re.match(
        r'^(?P<heat>\d+)\s+(?P<lane>\d+)\s+(?P<remarks>NM|DNS|DNF|DQ)$',
        tail_text,
        flags=re.I
    )
    if m:
        return {
            'RESULT': '',
            'HEAT': m.group('heat'),
            'LANE': m.group('lane'),
            'REMARKS': m.group('remarks').upper()
        }

    # Normal rows: result heat lane [remarks]
    m = re.match(
        rf'^(?P<result>{result_pattern})\s+'
        r'(?P<heat>\d+)\s+'
        r'(?P<lane>\d+)'
        r'(?:\s+(?P<remarks>[A-Za-z/]+))?$',
        tail_text,
        flags=re.I
    )
    if m:
        return {
            'RESULT': m.group('result'),
            'HEAT': m.group('heat'),
            'LANE': m.group('lane'),
            'REMARKS': (m.group('remarks') or '').strip()
        }

    return None


def parse_joined_row_text(joined: str):
    """
    Parse a fully flattened athlete row.
    Supports:
    - ranked normal rows
    - ranked status rows
    - unranked status rows
    """
    joined = re.sub(r'\s+', ' ', joined).strip()
    if not joined:
        return None

    # Ranked status row: rank tag body heat lane status
    m = re.match(
        r'^(?P<rank>\d+)\s+(?P<tag>\d+)\s+(?P<body>.+?)\s+(?P<heat>\d+)\s+(?P<lane>\d+)\s+(?P<remarks>NM|DNS|DNF|DQ)$',
        joined,
        flags=re.I
    )
    if m:
        name, team = split_name_team(m.group('body'))
        return {
            'RANK': m.group('rank'),
            'TAG_ID': m.group('tag'),
            'NAME': name,
            'TEAM': team,
            'RESULT': '',
            'HEAT': m.group('heat'),
            'LANE': m.group('lane'),
            'REMARKS': m.group('remarks').upper()
        }

    # Ranked normal row: rank tag body result heat lane [remarks]
    m = re.match(
        rf'^(?P<rank>\d+)\s+(?P<tag>\d+)\s+(?P<body>.+?)\s+'
        rf'(?P<result>{result_pattern})\s+(?P<heat>\d+)\s+(?P<lane>\d+)'
        r'(?:\s+(?P<remarks>[A-Za-z/]+))?$',
        joined,
        flags=re.I
    )
    if m:
        name, team = split_name_team(m.group('body'))
        return {
            'RANK': m.group('rank'),
            'TAG_ID': m.group('tag'),
            'NAME': name,
            'TEAM': team,
            'RESULT': m.group('result'),
            'HEAT': m.group('heat'),
            'LANE': m.group('lane'),
            'REMARKS': (m.group('remarks') or '').strip()
        }

    # Unranked status row: tag body heat lane status
    m = re.match(
        r'^(?P<tag>\d+)\s+(?P<body>.+?)\s+(?P<heat>\d+)\s+(?P<lane>\d+)\s+(?P<remarks>NM|DNS|DNF|DQ)$',
        joined,
        flags=re.I
    )
    if m:
        name, team = split_name_team(m.group('body'))
        return {
            'RANK': '',
            'TAG_ID': m.group('tag'),
            'NAME': name,
            'TEAM': team,
            'RESULT': '',
            'HEAT': m.group('heat'),
            'LANE': m.group('lane'),
            'REMARKS': m.group('remarks').upper()
        }

    return None


def parse_relay_joined_row_text(joined: str):
    """
    Parse relay/team rows:
    rank + school/team name + team code + result + heat + lane + optional remarks
    or unranked DNS/DQ rows with no numeric result.
    """
    joined = re.sub(r'\s+', ' ', joined).strip()
    if not joined:
        return None

    # Ranked relay row with result
    m = re.match(
        rf'^(?P<rank>\d+)\s+(?P<body>.+?)\s+'
        rf'(?P<result>{result_pattern})\s+(?P<heat>\d+)\s+(?P<lane>\d+)'
        r'(?:\s+(?P<remarks>[A-Za-z/]+))?$',
        joined,
        flags=re.I
    )
    if m:
        body = m.group('body').strip()
        parts = body.split()
        team = parts[-1] if parts else ''
        name = ' '.join(parts[:-1]).strip() if len(parts) > 1 else body

        return {
            'RANK': m.group('rank'),
            'TAG_ID': '',
            'NAME': name,
            'TEAM': team,
            'RESULT': m.group('result'),
            'HEAT': m.group('heat'),
            'LANE': m.group('lane'),
            'REMARKS': (m.group('remarks') or '').strip()
        }

    # Unranked relay status row
    m = re.match(
        r'^(?P<body>.+?)\s+(?P<heat>\d+)\s+(?P<lane>\d+)\s+(?P<remarks>NM|DNS|DNF|DQ)$',
        joined,
        flags=re.I
    )
    if m:
        body = m.group('body').strip()
        parts = body.split()
        team = parts[-1] if parts else ''
        name = ' '.join(parts[:-1]).strip() if len(parts) > 1 else body

        return {
            'RANK': '',
            'TAG_ID': '',
            'NAME': name,
            'TEAM': team,
            'RESULT': '',
            'HEAT': m.group('heat'),
            'LANE': m.group('lane'),
            'REMARKS': m.group('remarks').upper()
        }

    return None


def parse_structured_cells(cells):
    """
    Prefer structured cell parsing when possible.
    """
    # Ranked athlete row: rank, tag, name, team..., tail...
    if len(cells) >= 5 and re.fullmatch(r'\d+', cells[0]) and re.fullmatch(r'\d+', cells[1]):
        rank = cells[0]
        tag_id = cells[1]
        name_cell = cells[2].strip()

        for team_width in [1, 2, 3, 4]:
            team_end = 3 + team_width
            if team_end > len(cells):
                continue

            team_candidate = ' '.join([c for c in cells[3:team_end] if c]).strip()
            tail_text = ' '.join([c for c in cells[team_end:] if c]).strip()

            if not team_candidate:
                continue

            tail = parse_tail_from_text(tail_text)
            if tail:
                return {
                    'RANK': rank,
                    'TAG_ID': tag_id,
                    'NAME': name_cell,
                    'TEAM': team_candidate,
                    'RESULT': tail['RESULT'],
                    'HEAT': tail['HEAT'],
                    'LANE': tail['LANE'],
                    'REMARKS': tail['REMARKS']
                }

    # Unranked athlete status row: tag, name, team..., heat, lane, status
    if len(cells) >= 4 and re.fullmatch(r'\d+', cells[0]) and not re.fullmatch(r'\d+', cells[1]):
        tag_id = cells[0]
        name_cell = cells[1].strip()

        for team_width in [1, 2, 3, 4]:
            team_end = 2 + team_width
            if team_end > len(cells):
                continue

            team_candidate = ' '.join([c for c in cells[2:team_end] if c]).strip()
            tail_text = ' '.join([c for c in cells[team_end:] if c]).strip()

            if not team_candidate:
                continue

            tail = parse_tail_from_text(tail_text)
            if tail and tail['RESULT'] == '':
                return {
                    'RANK': '',
                    'TAG_ID': tag_id,
                    'NAME': name_cell,
                    'TEAM': team_candidate,
                    'RESULT': '',
                    'HEAT': tail['HEAT'],
                    'LANE': tail['LANE'],
                    'REMARKS': tail['REMARKS']
                }

    return None


def is_tail_only_fragment(text: str) -> bool:
    """
    Detect orphan fragments like:
    - 63 5 12 q
    - 04 2 3 q
    - 44 1 1
    - 96 2 6 q
    """
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return False

    patterns = [
        r'^(?:\d{1,2}:\d{2}\.)?\d+\s+\d+\s+\d+(?:\s+[A-Za-z/]+)?$',
        r'^\d+\s+\d+\s+(?:NM|DNS|DNF|DQ)$'
    ]
    return any(re.fullmatch(p, text, flags=re.I) for p in patterns)


def row_to_record(raw_row):
    """
    Parse athlete and relay rows robustly.
    """
    cells = [clean_cell(x) for x in raw_row]
    cells = [c for c in cells if c != '']

    if not cells:
        return None

    joined = re.sub(r'\s+', ' ', ' '.join(cells)).strip()

    if is_junk_row([joined]):
        return None

    rec = parse_structured_cells(cells)
    if rec is not None:
        return rec

    rec = parse_joined_row_text(joined)
    if rec is not None:
        return rec

    rec = parse_relay_joined_row_text(joined)
    if rec is not None:
        return rec

    return None


def repair_split_rows(table):
    """
    Repair/ignore split rows before parsing.
    """
    if not table:
        return []

    repaired = []

    for raw_row in table:
        cells = [clean_cell(x) for x in raw_row]
        cells = [c for c in cells if c != '']

        if not cells:
            continue

        joined = re.sub(r'\s+', ' ', ' '.join(cells)).strip()

        if is_junk_row([joined]):
            repaired.append(cells)
            continue

        current_parseable = row_to_record(cells) is not None

        if not repaired:
            repaired.append(cells)
            continue

        prev_cells = repaired[-1]
        prev_parseable = row_to_record(prev_cells) is not None

        if current_parseable:
            repaired.append(cells)
            continue

        if is_tail_only_fragment(joined):
            if not prev_parseable:
                repaired[-1] = prev_cells + cells
            else:
                continue
        else:
            if not prev_parseable:
                repaired[-1] = prev_cells + cells
            else:
                repaired.append(cells)

    return repaired


def looks_like_athlete_line(line: str) -> bool:
    """
    Detect likely athlete or relay result lines from extracted text.
    """
    line = re.sub(r'\s+', ' ', line).strip()
    if not line:
        return False

    # Ranked athlete normal row
    if re.match(
        rf'^\d+\s+\d+\s+.+\s+{result_pattern}\s+\d+\s+\d+(?:\s+[A-Za-z/]+)?$',
        line,
        flags=re.I
    ):
        return True

    # Ranked athlete status row
    if re.match(
        r'^\d+\s+\d+\s+.+\s+\d+\s+\d+\s+(?:NM|DNS|DNF|DQ)$',
        line,
        flags=re.I
    ):
        return True

    # Unranked athlete status row
    if re.match(
        r'^\d+\s+.+\s+\d+\s+\d+\s+(?:NM|DNS|DNF|DQ)$',
        line,
        flags=re.I
    ):
        return True

    # Ranked relay row
    if re.match(
        rf'^\d+\s+.+\s+{result_pattern}\s+\d+\s+\d+(?:\s+[A-Za-z/]+)?$',
        line,
        flags=re.I
    ):
        return True

    # Unranked relay status row
    if re.match(
        r'^.+\s+\d+\s+\d+\s+(?:NM|DNS|DNF|DQ)$',
        line,
        flags=re.I
    ):
        return True

    return False


def build_page_dataframe_from_text(text: str):
    """
    Fallback parser using page text lines when table extraction fails.
    """
    records = []
    lines = [re.sub(r'\s+', ' ', ln).strip() for ln in text.splitlines() if ln.strip()]

    for line in lines:
        if is_junk_row([line]):
            continue
        if is_tail_only_fragment(line):
            continue
        if not looks_like_athlete_line(line):
            continue

        rec = parse_joined_row_text(line)
        if rec is None:
            rec = parse_relay_joined_row_text(line)

        if rec is not None:
            records.append(rec)

    if not records:
        return pd.DataFrame(columns=['RANK', 'TAG_ID', 'NAME', 'TEAM', 'RESULT', 'HEAT', 'LANE', 'REMARKS'])

    return pd.DataFrame(records)


def build_page_dataframe(table, text):
    """
    First try table-based parsing. If that yields nothing,
    fall back to direct text-line parsing.
    """
    if table:
        repaired_rows = repair_split_rows(table)
        records = []
        for raw_row in repaired_rows:
            rec = row_to_record(raw_row)
            if rec is not None:
                records.append(rec)

        if records:
            return pd.DataFrame(records)

    return build_page_dataframe_from_text(text)


def apply_result_and_qualification_mapping(df):
    if df.empty:
        return df

    df['REMARKS'] = df['REMARKS'].fillna('').astype(str)
    df['RESULT'] = df['RESULT'].fillna('').astype(str)
    df['QUALIFICATION'] = df['QUALIFICATION'].fillna('').astype(str)

    for code in ['NM', 'DNS', 'DQ', 'DNF']:
        mask = df['REMARKS'].str.contains(rf'\b{code}\b', na=False, regex=True)
        df.loc[mask, 'RESULT'] = code

    mask = df['REMARKS'].str.fullmatch(r'q', na=False)
    df.loc[mask, 'QUALIFICATION'] = 'q'

    mask = df['REMARKS'].str.fullmatch(r'Q', na=False)
    df.loc[mask, 'QUALIFICATION'] = 'Q'

    return df


# ---------------------------------------------------------
# Main extraction
# ---------------------------------------------------------
df_table = pd.DataFrame()

current_meta = {
    'DATE': '',
    'EVENT': '',
    'DIVISION': '',
    'STAGE': '',
    'GENDER': '',
    'SUB_EVENT': '',
    'SESSION': '',
    'EVENT_CLASS': '',
    'CATEGORY_EVENT': ''
}

current_event_key = None
wind_dict = {}
event_has_explicit_wind = False

with pdfplumber.open(file) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        text = page.extract_text() or ''
        if not text.strip():
            continue

        page_meta = parse_page_metadata(text, current_meta)

        new_event_key = (
            page_meta.get('EVENT', ''),
            page_meta.get('DIVISION', ''),
            page_meta.get('STAGE', ''),
            page_meta.get('SESSION', '')
        )

        if new_event_key != current_event_key:
            wind_dict = {}
            event_has_explicit_wind = False
            current_event_key = new_event_key

        page_wind, page_has_explicit_wind = extract_explicit_wind_dict(text)
        if page_has_explicit_wind:
            event_has_explicit_wind = True
            wind_dict.update(page_wind)

        table = extract_table_with_fallback(page)
        page_df = build_page_dataframe(table, text)

        if page_df.empty:
            current_meta = page_meta
            continue

        page_df['FIRST_NAME'] = ''
        page_df['LAST_NAME'] = ''
        page_df['OTHER_NAME'] = ''
        page_df['DISTANCE'] = ''
        page_df['EVENT'] = page_meta.get('EVENT', '')
        page_df['GENDER'] = page_meta.get('GENDER', '')
        page_df['DIVISION'] = page_meta.get('DIVISION', '')
        page_df['STAGE'] = page_meta.get('STAGE', '')
        page_df['COMPETITION'] = 'National School Games'
        page_df['REGION'] = 'Local'
        page_df['CATEGORY_EVENT'] = page_meta.get('CATEGORY_EVENT', '')
        page_df['SOURCE'] = SOURCE_URL
        page_df['YEAR'] = '2026'
        page_df['QUALIFICATION'] = ''
        page_df['POINTS'] = ''
        page_df['UNIQUE_ID'] = ''
        page_df['NATIONALITY'] = ''
        page_df['DICT_RESULTS'] = ''
        page_df['DOB'] = ''
        page_df['GROUP'] = ''
        page_df['ATHLETE_ID'] = ''
        page_df['TIMESTAMP'] = ''
        page_df['VENUE'] = ''
        page_df['DATE'] = page_meta.get('DATE', '')
        page_df['SUB_EVENT'] = page_meta.get('SUB_EVENT', '')
        page_df['SESSION'] = page_meta.get('SESSION', '')
        page_df['EVENT_CLASS'] = page_meta.get('EVENT_CLASS', '')
        page_df['HOST_CITY'] = 'Singapore'
        page_df['RX_TIME'] = ''
        page_df['SEED'] = ''
        page_df['AGE'] = ''

        if event_has_explicit_wind:
            page_df['WIND'] = page_df['HEAT'].apply(
                lambda x: wind_dict.get(f'H{str(x).strip()}', '') if str(x).strip() else ''
            )
        else:
            page_df['WIND'] = ''

        mask_para = page_df['EVENT'].str.contains(r'Para', case=False, na=False)
        page_df.loc[mask_para, 'SUB_EVENT'] = 'Para'

        page_df = apply_result_and_qualification_mapping(page_df)

        df_table = pd.concat([df_table, page_df], axis=0, ignore_index=True)

        current_meta = page_meta


# ---------------------------------------------------------
# Final schema
# ---------------------------------------------------------
for col in FINAL_SCHEMA:
    if col not in df_table.columns:
        df_table[col] = ''

df_table = df_table.reindex(columns=FINAL_SCHEMA)

for col in df_table.columns:
    df_table[col] = df_table[col].astype(str).str.strip()

df_table = df_table.replace('nan', '')
df_table = df_table.replace('None', '')

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
output_file = 'sectrack_result_2026a_patched.csv'
df_table.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Done. Rows extracted: {len(df_table)}")
print(f"Saved to: {output_file}")
print(df_table.head(20))

# Optional quick sanity checks
print(sorted([x for x in df_table['EVENT_CLASS'].dropna().unique() if x]))
print(sorted([x for x in df_table['GENDER'].dropna().unique() if x]))
print(sorted([x for x in df_table['DIVISION'].dropna().unique() if x]))

Done. Rows extracted: 4739
Saved to: sectrack_result_2026a_patched.csv
   FIRST_NAME LAST_NAME OTHER_NAME  \
0                                    
1                                    
2                                    
3                                    
4                                    
5                                    
6                                    
7                                    
8                                    
9                                    
10                                   
11                                   
12                                   
13                                   
14                                   
15                                   
16                                   
17                                   
18                                   
19                                   

                                                NAME RANK TAG_ID    TEAM SEED  \
0                 Joshua Lee Shyen ACS (Independe

In [6]:
df_table

,FIRST_NAME,LAST_NAME,OTHER_NAME,NAME,RANK,TAG_ID,TEAM,SEED,RESULT,QUALIFICATION,...,SOURCE,REMARKS,TIMESTAMP,VENUE,SUB_EVENT,SESSION,EVENT_CLASS,DISTANCE,HOST_CITY,RX_TIME
0,,,,Joshua Lee Shyen ACS (Independent),1,53,ACS(I),,7.25,,...,https://nsg.rjcat.com/sectrack_2026/ss_home.ph...,,,,,S01-01,,,Singapore,
1,,,,"Chua Je-An, Garrett Raffles Institution",2,107,RI,,6.79,,...,https://nsg.rjcat.com/sectrack_2026/ss_home.ph...,,,,,S01-01,,,Singapore,
2,,,,"Tang Kai Sheng, Cayman Victoria Junior College",3,213,VJC,,6.45,,...,https://nsg.rjcat.com/sectrack_2026/ss_home.ph...,,,,,S01-01,,,Singapore,
3,,,,Chang I Teng Dunman High School,4,173,DHS,,6.20,,...,https://nsg.rjcat.com/sectrack_2026/ss_home.ph...,,,,,S01-01,,,Singapore,
4,,,,Lim Kyan Victoria Junior College,5,201,VJC,,6.04,,...,https://nsg.rjcat.com/sectrack_2026/ss_home.ph...,,,,,S01-01,,,Singapore,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4734,,,,National Junior College National Junior College,5,,NJC,,03:30.27,,...,https://nsg.rjcat.com/sectrack_2026/ss_home.ph...,,,,,S21-16,,,Singapore,
4735,,,,Eunoia Junior College Eunoia Junior College,6,,EJC,,03:30.39,,...,https://nsg.rjcat.com/sectrack_2026/ss_home.ph...,,,,,S21-16,,,Singapore,
4736,,,,Dunman High School Dunman High School,7,,DHS,,03:44.76,,...,https://nsg.rjcat.com/sectrack_2026/ss_home.ph...,,,,,S21-16,,,Singapore,
4737,,,,Catholic Junior College Catholic Junior College,8,,CJC,,03:44.84,,...,https://nsg.rjcat.com/sectrack_2026/ss_home.ph...,,,,,S21-16,,,Singapore,


In [7]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/NSG/2026/Consolidated')

df_table.to_csv('sectrack_result_2026a.csv', sep=',', index=False, encoding='utf-8')

In [8]:
def clean_name(name): # remove long name of school
    
    list=[]
    
 #   sschool_html = pd.read_html("https://en.wikipedia.org/wiki/List_of_secondary_schools_in_Singapore")
    
 #   sec_schools=sschool_html[0]['Name'].tolist()  # put schools into a list
        
 #   for i in range(len(sec_schools)): # extract first word from school to match        
        
 #       list.append(sec_schools[i].split()[0])

    # Read the Excel file
    os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Singapore Secondary Schools')

    df = pd.read_excel("secondary_schools_singapore.xlsx", sheet_name="Secondary Schools")

    # First column contains the school names
    sec_schools = df.iloc[:, 0].dropna().astype(str).tolist()

    # Extract first word from each school name
    for school in sec_schools:
        list.append(school.split()[0])
     
    list.remove("St.")
    list.remove("Yuan")
    list.append("SJI")
    list.append("ACS")
    list.append("Dunman")
    list.append("Crescent")
    list.append("Raffles")
    list.append("NorthLight")
    list.append("Catholic")
    list.append("Methodist")
    list.append("NUS")
    list.append("Bukit")
    list.append("Riverside")
    list.append("River")
    list.append("Fairfield")
    list.append("Admiralty")
    list.append("Cedar")
    list.append("Outram")
    list.append("Bedok")
    list.append("Nan Chiau")
    list.append("Hua Yi")
    list.append("Pasir")
    list.append("Nanyang")
    list.append("Junyuan")
    list.append("Crest")
    list.append("Eunoia")
    list.append("North")
    list.append("Katong")
    list.append("Queensway")
    list.append("Orchid")
    list.append("Evergreen")
    list.append("Bendemeer")
    list.append("Beatty")
    list.append("Bartley")
    list.append("Guangyang")
    list.append("Jurong")
    list.append("Ahmad Ibrahim")
    list.append("Deyi")
    list.append("Kranji")
    list.append("CHIJ")
    list.append("Paya")
    list.append("Gan Eng")
    list.append("Changkat")
    list.append("Chung Cheng")
    list.append("Pathlight")
    list.append("Anglo-Chinese")
    list.append("Millennia")
    list.append("Tanjong")
    list.append("Manjusri")
    list.append("Edgefield")
    list.append("Anglican")
    list.append("St. ")
    list.append("Aahnmad")
    list.append("CHIJ St.")
    list.append("Assumption")
    list.append("Geylang")
    list.append("Yang Seng Kang")
    list.append("Montfort")
    list.append("Mayflower")
    list.append("Hwa Chong")
    list.append("Anderson Sec")
    list.append("FuHua")
    list.append("Nan Hua")
    list.append("Commonwealth")
    list.append("HasBbuukllit")
    list.append("SP-CCA")
    list.append("Maris Stella")
    list.append("Broadrick")
    list.append("Spectra")
    list.append("Singapore")
    list.append("Clementi")
    list.append("Regent")
    list.append("Peirce")
    list.append("ReSdinzgwapaonre")
    list.append("CNhaaznrgkat")
    list.append("KhCaihraingkat")
    list.append("Yuan Ching Sec")
    list.append("Fuhua Sec")
    list.append("Grace Orchard")
    list.append("Kent Ridge Sec")
    list.append("Greendale Sec")
    list.append("Compassvale Sec")
    list.append("Hillgrove Sec")
    list.append("Presbyterian High")
    list.append("Rainbow Centre")
    list.append("AWWA")
    list.append("APSN")
    list.append("Meridian Sec")
    list.append("Peicai Sec")



     





    
    cleaned_name=''
    
    #if 'Sec' in name:
        
    #    name = name.replace("Sec", "Secondary")

    
    
    for i in range(len(list)):
                

        if re.search(list[i], name):
            
            if name.count(list[i])==1: # athlete name + school name concatenated
                
                pos = re.search(list[i], name)

                split_end=pos.start()

                cleaned_name=name[:split_end]
                                     
            elif name.count(list[i])==2: # if school name pattern is repeated
                                     
                x = name.find(list[i])
                
                if x != -1:  # -1 means not found
                    
                    y = name.find(list[i], x+1)
                    
                    if y != -1:
                        
                        cleaned_name=name[:y]                

        
    
    return cleaned_name
                

    
    

def separate(string):
    
    print(string)
    try:
    
        components=string.split(' ')
        qualification = ''
    
        if components[-1]=='Q' or components[-1]=='q':
        
            qualification=components[-1]
            
    except:
        
        qualification = ''
            
    
    return qualification
    
def strip(string):
    
    try:
    
        components=string.split(' ')
        qualification = ''
    
        new=''
    
        if components[-1]=='Q' or components[-1]=='q':
        
            popped = components.pop()
        
            new=' '.join(components)
        
        else:
        
            new=string
            
    except:
        
        new=string
            
    
    return new

# Extract event class

def event_class(string):
    
    
    try:
        
        left=re.search(r'\(', string)
        right=re.search(r'\)', string)
        
        
        type = string[left.start()+1:right.start()]
        
                 
    except:
        
        type = ''
    
    return type
    
def clean_event(string): # clean event from brackets and strip out PARA from event
    
    
    try:
        
        left=re.search(r'\(', string)
        right=re.search(r'\)', string)
        
        
        event_stripped = string[:left.start()]
        
        print(event_stripped)
        
                 
    except:
        
        event_stripped = string
        
    
    return event_stripped

def remarks(string): # clean NM, DNS and other junk from remarks column
    
    
    try:
        
        junk = re.search(r'NM\s|DNS\s|DQ\s|DNF\s', string)        
        
        cleaned = string[junk.end():]
        
                 
    except:
        
        cleaned = string
    
    return cleaned

def split_name(string):
    
    midpoint=len(string)//2
    
    return string[:midpoint]


for col in df_table.columns:
    df_table[col] = df_table[col].astype(str)
    df_table[col] = df_table[col].str.replace('\xa0', ' ', regex=True)
    df_table[col] = df_table[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    df_table[col] = df_table[col].str.replace('\r', ' ', regex=True)
    df_table[col] = df_table[col].str.replace('\n', ' ', regex=True)
    df_table[col] = df_table[col].str.strip()
    df_table['NAME'] = df_table['NAME'].apply(str)

#df_table['EVENT_CLASS'] = df_table['EVENT'].apply(event_class)
#df_table['QUALIFICATION']=df_table['NAME'].apply(separate)
df_table['NAME_clean']=df_table['NAME'].apply(clean_name)
df_table['EVENT'] = df_table['EVENT'].apply(clean_event)
df_table['REMARKS']=df_table['REMARKS'].apply(remarks)

df_table=df_table[df_table['NAME']!='None']
    

#df_table = df_table.reindex(columns= ['FIRST_NAME', 'LAST_NAME', 'OTHER_NAME', 'NAME', 'RANK', 'TAG_ID', 'TEAM', 'SEED', 'RESULT', 'QUALIFICATION',
#                                        'HEAT', 'LANE', 'WIND', 'EVENT', 'DIVISION', 'STAGE', 'POINTS', 'AGE', 'GENDER', 'UNIQUE_ID', 'NATIONALITY',
#                                        'DICT_RESULTS', 'YEAR', 'DATE', 'COMPETITION', 'REGION', 'DOB', 'GROUP', 'CATEGORY_EVENT', 'ATHLETE_ID',
#                                        'SOURCE', 'REMARKS', 'TIMESTAMP', 'VENUE', 'SUB_EVENT', 'SESSION', 'EVENT_CLASS', 'DISTANCE', 'HOST_CITY', 'RX_TIME'])

os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/NSG/2026/')

df_table.to_csv('test_output.csv', sep=',', index=False, encoding='utf-8')

Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Long Jump 
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
Shot Put  
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m 
1500m

In [9]:
df_table.columns

Index(['FIRST_NAME', 'LAST_NAME', 'OTHER_NAME', 'NAME', 'RANK', 'TAG_ID',
       'TEAM', 'SEED', 'RESULT', 'QUALIFICATION', 'HEAT', 'LANE', 'WIND',
       'EVENT', 'DIVISION', 'STAGE', 'POINTS', 'AGE', 'GENDER', 'UNIQUE_ID',
       'NATIONALITY', 'DICT_RESULTS', 'YEAR', 'DATE', 'COMPETITION', 'REGION',
       'DOB', 'GROUP', 'CATEGORY_EVENT', 'ATHLETE_ID', 'SOURCE', 'REMARKS',
       'TIMESTAMP', 'VENUE', 'SUB_EVENT', 'SESSION', 'EVENT_CLASS', 'DISTANCE',
       'HOST_CITY', 'RX_TIME', 'NAME_clean'],
      dtype='object')

In [131]:
df_table['NAME'] = df_table['NAME_clean']
df_table = df_table.drop(columns=['NAME_clean'])


df_table = df_table.reindex(columns= ['FIRST_NAME', 'LAST_NAME', 'OTHER_NAME', 'NAME', 'RANK', 'TAG_ID', 'TEAM', 'SEED', 'RESULT', 'QUALIFICATION',
                                        'HEAT', 'LANE', 'WIND', 'EVENT', 'DIVISION', 'STAGE', 'POINTS', 'AGE', 'GENDER', 'UNIQUE_ID', 'NATIONALITY',
                                        'DICT_RESULTS', 'YEAR', 'DATE', 'COMPETITION', 'REGION', 'DOB', 'GROUP', 'CATEGORY_EVENT', 'ATHLETE_ID',
                                        'SOURCE', 'REMARKS', 'TIMESTAMP', 'VENUE', 'SUB_EVENT', 'SESSION', 'EVENT_CLASS', 'DISTANCE', 'HOST_CITY', 'RX_TIME'])

os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/NSG/2026/Consolidated')

df_table.to_excel('sectrack_result_2026a.xlsx', index=False)